In [ ]:
# -*- coding: utf-8 -*-
"""rpi_irrigation_by_evaporation.py

ET-based irrigation decision (FAO-56 Penman–Monteith daily ET0).
- Prints a full log line every minute to the terminal
- Makes ONE decision per day at 19:00 (local time)
- Sends the decision + open seconds to ThingSpeak command channel
- Rain handling supports BOTH incremental and cumulative gauges (AUTO mode)
"""

import time
import math
import requests
from datetime import datetime
from zoneinfo import ZoneInfo

# =========================
# CONFIG (YOUR SETUP)
# =========================

SENSOR_CHANNEL_ID = "3229286"
SENSOR_READ_APIKEY = "CDFQZVMUYDNXPFVY"

CMD_WRITE_APIKEY = "O9KA0TYQC4NW0L3S"

FIELD_T = "field1"      # Temperature (°C)
FIELD_RS = "field2"     # Solar radiation (W/m²)
FIELD_WIND = "field3"   # Wind speed (m/s)
FIELD_RH = "field4"     # Relative humidity (%)
FIELD_RAIN = "field5"   # Rain (mm, incremental)

DT_MIN = 10
PY_LOOP_SEC = 60

Kc = 1.0
THRESH_MM = 2.5

# --- Irrigation system (one big pipe, no drippers) ---
# IMPORTANT: Set FLOW_LPH to your real pipe flow (liters per hour).
# If you know liters per minute, multiply by 60.
FLOW_LPH = 240.0          # liters/hour (CHANGE THIS to your real value)
WETTED_AREA_M2 = 1.5      # m^2 (e.g., 0.5 * 3.0 = 1.5)

# Convert to mm/hour (1 mm over 1 m^2 = 1 liter)
MM_PER_HOUR = FLOW_LPH / max(1e-9, WETTED_AREA_M2)

MIN_OPEN_SECONDS = 30
MAX_OPEN_SECONDS = 1800

LOCAL_TZ = "Asia/Jerusalem"
DECISION_HOUR = 19
DECISION_MINUTE = 0

LAT_DEG = 31.9
ELEV_M = 70.0

CMD_FIELD_CMD_ID = "field1"
CMD_FIELD_IRRIGATE = "field2"
CMD_FIELD_OPEN_SEC = "field3"
CMD_FIELD_ET0 = "field4"
CMD_FIELD_CUM_ETC = "field5"

# Rain gauge mode:
# "auto"        -> supports both incremental and cumulative
# "incremental" -> field is mm per interval
# "cumulative"  -> field is cumulative mm since day start (or since reset)
RAIN_MODE = "auto"

# =========================
# ThingSpeak helpers
# =========================

def ts_read_last(channel_id: str, read_api_key: str = "") -> dict:
    url = f"https://api.thingspeak.com/channels/{channel_id}/feeds/last.json"
    params = {}
    if read_api_key:
        params["api_key"] = read_api_key
    r = requests.get(url, params=params, timeout=10)
    r.raise_for_status()
    return r.json()

def ts_write(write_api_key: str, fields: dict) -> None:
    url = "https://api.thingspeak.com/update.json"
    params = {"api_key": write_api_key}
    params.update(fields)
    r = requests.get(url, params=params, timeout=10)
    r.raise_for_status()

def safe_float(x, default=0.0):
    try:
        if x is None or x == "":
            return default
        return float(x)
    except Exception:
        return default

# =========================
# Rain handling (AUTO supports incremental + cumulative)
# =========================

def rain_delta_mm(rain_raw: float,
                  prev_rain_raw,
                  now_local: datetime,
                  mode: str,
                  inferred_mode: str):
    """
    Returns:
      delta_mm (float)   - rain to add for this sample (mm)
      new_prev (float)   - updated prev_rain_raw
      new_inferred_mode  - inferred mode ("cumulative" or "incremental")

    Logic:
      - incremental: delta = rain_raw
      - cumulative:  delta = max(rain_raw - prev_rain_raw, 0)
      - auto: start safe as cumulative (prevents overcount), switch to incremental
              if we observe a drop that is NOT a daily reset.
    """
    rain_raw = max(0.0, float(rain_raw))

    # manual override
    if mode == "incremental":
        return rain_raw, rain_raw, "incremental"

    if mode == "cumulative":
        if prev_rain_raw is None:
            return rain_raw, rain_raw, "cumulative"
        return max(rain_raw - prev_rain_raw, 0.0), rain_raw, "cumulative"

    # AUTO mode
    if prev_rain_raw is None:
        # start with whatever inferred mode is set to
        if inferred_mode not in ("cumulative", "incremental"):
            inferred_mode = "cumulative"
        # if incremental, first value is delta; if cumulative, first value is cumulative (ok)
        if inferred_mode == "incremental":
            return rain_raw, rain_raw, "incremental"
        return rain_raw, rain_raw, "cumulative"

    # If we are treating as cumulative and the value drops, it can be:
    # - daily reset near midnight (still cumulative)
    # - not cumulative (switch to incremental)
    if inferred_mode == "cumulative":
        if rain_raw < prev_rain_raw - 1e-6:
            if rain_raw <= 0.01 and now_local.hour <= 2:
                # daily reset at night
                return 0.0, rain_raw, "cumulative"
            else:
                # drop at other time -> likely incremental
                inferred_mode = "incremental"

    if inferred_mode == "incremental":
        return rain_raw, rain_raw, "incremental"

    # still cumulative
    return max(rain_raw - prev_rain_raw, 0.0), rain_raw, "cumulative"

# =========================
# FAO-56 Penman–Monteith (daily ET0)
# =========================

def sat_vapor_pressure_kpa(t_c: float) -> float:
    return 0.6108 * math.exp((17.27 * t_c) / (t_c + 237.3))

def slope_vapor_pressure_curve_kpa_per_c(t_c: float) -> float:
    es = sat_vapor_pressure_kpa(t_c)
    return 4098.0 * es / ((t_c + 237.3) ** 2)

def atm_pressure_kpa(elev_m: float) -> float:
    return 101.3 * (((293.0 - 0.0065 * elev_m) / 293.0) ** 5.26)

def psychrometric_constant_kpa_per_c(p_kpa: float) -> float:
    return 0.000665 * p_kpa

def extraterrestrial_radiation_ra_mj_m2_day(lat_deg: float, doy: int) -> float:
    lat_rad = math.radians(lat_deg)
    dr = 1.0 + 0.033 * math.cos(2.0 * math.pi * doy / 365.0)
    delta = 0.409 * math.sin(2.0 * math.pi * doy / 365.0 - 1.39)
    x = -math.tan(lat_rad) * math.tan(delta)
    x = max(-1.0, min(1.0, x))
    ws = math.acos(x)
    Gsc = 0.0820
    ra = (24.0 * 60.0 / math.pi) * Gsc * dr * (
        ws * math.sin(lat_rad) * math.sin(delta) +
        math.cos(lat_rad) * math.cos(delta) * math.sin(ws)
    )
    return ra

def et0_fao56_pm_daily(t_c_mean: float, rh_mean_pct: float, u2_mean: float,
                      rs_mj_m2_day: float, lat_deg: float, doy: int, elev_m: float) -> float:
    es = sat_vapor_pressure_kpa(t_c_mean)
    ea = (max(0.0, min(100.0, rh_mean_pct)) / 100.0) * es
    vpd = max(0.0, es - ea)

    delta = slope_vapor_pressure_curve_kpa_per_c(t_c_mean)
    p_kpa = atm_pressure_kpa(elev_m)
    gamma = psychrometric_constant_kpa_per_c(p_kpa)

    rs = max(0.0, rs_mj_m2_day)
    albedo = 0.23
    rns = (1.0 - albedo) * rs

    ra = extraterrestrial_radiation_ra_mj_m2_day(lat_deg, doy)
    rso = (0.75 + 2e-5 * elev_m) * ra
    rs_rso = rs / max(1e-6, rso)

    sigma = 4.903e-9
    t_k = t_c_mean + 273.16
    rnl = sigma * (t_k ** 4) * (0.34 - 0.14 * math.sqrt(max(ea, 0.0))) * (1.35 * min(rs_rso, 1.0) - 0.35)
    rnl = max(0.0, rnl)

    rn = max(0.0, rns - rnl)
    g = 0.0

    u2 = max(0.0, u2_mean)
    num = 0.408 * delta * (rn - g) + gamma * (900.0 / (t_c_mean + 273.0)) * u2 * vpd
    den = delta + gamma * (1.0 + 0.34 * u2)
    return max(0.0, num / max(1e-9, den))

# =========================
# MAIN
# =========================

def main():
    print("Running... (minute logs; daily decision at 19:00; sends command to ThingSpeak)")
    tz = ZoneInfo(LOCAL_TZ)

    sum_t = sum_rh = sum_wind = 0.0
    n_samples = 0
    rs_j_m2 = 0.0
    rain_mm_day = 0.0

    cmd_id = 0
    last_created_at = None
    last_local_time = None
    last_decision_date = None

    prev_rain_raw = None
    rain_mode_inferred = "cumulative"  # AUTO starts safe

    while True:
        try:
            now_local = datetime.now(tz)

            last = ts_read_last(SENSOR_CHANNEL_ID, SENSOR_READ_APIKEY)
            created_at = last.get("created_at")

            if not created_at:
                print(f"[{now_local.strftime('%Y-%m-%d %H:%M:%S')}] No created_at in ThingSpeak response")
                time.sleep(PY_LOOP_SEC)
                continue

            # If no new sample, still print a heartbeat every minute
            if created_at == last_created_at:
                print(f"[{now_local.strftime('%Y-%m-%d %H:%M:%S')}] No new ThingSpeak data (waiting)...")
                time.sleep(PY_LOOP_SEC)
                continue
            last_created_at = created_at

            t_c = safe_float(last.get(FIELD_T), 0.0)
            rh = safe_float(last.get(FIELD_RH), 0.0)
            rs = safe_float(last.get(FIELD_RS), 0.0)       # W/m2
            wind = safe_float(last.get(FIELD_WIND), 0.0)   # m/s
            rain_raw = safe_float(last.get(FIELD_RAIN), 0.0)

            # Fix RH if sensor sends 0-1 scale
            if rh <= 1.5:
                rh *= 100.0

            # dt for integration
            if last_local_time is None:
                dt_s = DT_MIN * 60
            else:
                dt_s = (now_local - last_local_time).total_seconds()
                if dt_s <= 0 or dt_s > 3600:
                    dt_s = DT_MIN * 60
            last_local_time = now_local

            # Rain delta (auto supports cumulative/incremental)
            rain_mm, prev_rain_raw, rain_mode_inferred = rain_delta_mm(
                rain_raw=rain_raw,
                prev_rain_raw=prev_rain_raw,
                now_local=now_local,
                mode=RAIN_MODE,
                inferred_mode=rain_mode_inferred
            )

            # accumulate daily means + radiation integral + rain sum
            sum_t += t_c
            sum_rh += rh
            sum_wind += wind
            n_samples += 1

            rs_j_m2 += max(0.0, rs) * dt_s
            rain_mm_day += max(0.0, rain_mm)

            # Minute log (full line each minute)
            rs_mj_m2_so_far = rs_j_m2 / 1e6
            print(
                f"[{now_local.strftime('%Y-%m-%d %H:%M:%S')}] "
                f"T={t_c:.2f}C RH={rh:.1f}% Rs={rs:.1f}W/m2 u2={wind:.2f}m/s "
                f"RainRaw={rain_raw:.3f}mm RainAdd={rain_mm:.3f}mm Mode={rain_mode_inferred} dt={dt_s:.0f}s | "
                f"Samples={n_samples} Rs_cum={rs_mj_m2_so_far:.3f}MJ/m2 Rain_cum={rain_mm_day:.3f}mm"
            )

            # Daily decision at 19:00 local time
            if (now_local.hour == DECISION_HOUR and now_local.minute == DECISION_MINUTE
                and last_decision_date != now_local.date()):

                last_decision_date = now_local.date()

                t_mean = sum_t / max(1, n_samples)
                rh_mean = sum_rh / max(1, n_samples)
                wind_mean = sum_wind / max(1, n_samples)

                rs_mj_m2 = rs_j_m2 / 1e6
                doy = int(now_local.strftime("%j"))

                et0_mm_day = et0_fao56_pm_daily(
                    t_c_mean=t_mean,
                    rh_mean_pct=rh_mean,
                    u2_mean=wind_mean,
                    rs_mj_m2_day=rs_mj_m2,
                    lat_deg=LAT_DEG,
                    doy=doy,
                    elev_m=ELEV_M
                )

                etc_mm_day = et0_mm_day * Kc
                net_mm = max(etc_mm_day - rain_mm_day, 0.0)

                irrigate = 1 if net_mm >= THRESH_MM else 0

                if irrigate == 1:
                    open_sec = int(round((net_mm / MM_PER_HOUR) * 3600.0))
                    open_sec = max(MIN_OPEN_SECONDS, min(open_sec, MAX_OPEN_SECONDS))
                    cmd_id += 1
                else:
                    open_sec = 0

                payload = {
                    CMD_FIELD_CMD_ID: cmd_id,
                    CMD_FIELD_IRRIGATE: irrigate,
                    CMD_FIELD_OPEN_SEC: open_sec,
                    CMD_FIELD_ET0: round(et0_mm_day, 4),
                    CMD_FIELD_CUM_ETC: round(net_mm, 4),
                }
                ts_write(CMD_WRITE_APIKEY, payload)

                print(
                    f"[DECISION {now_local.strftime('%Y-%m-%d %H:%M')}] "
                    f"Tmean={t_mean:.2f}C RHmean={rh_mean:.1f}% u2mean={wind_mean:.2f}m/s "
                    f"Rs_day={rs_mj_m2:.3f}MJ/m2 Rain_day={rain_mm_day:.3f}mm "
                    f"| ET0={et0_mm_day:.3f} ETC={etc_mm_day:.3f} Net={net_mm:.3f} "
                    f"| irrigate={irrigate} open_sec={open_sec} cmd_id={cmd_id} "
                    f"| sent_to_thingspeak=YES"
                )

                # reset accumulators for next day
                sum_t = sum_rh = sum_wind = 0.0
                n_samples = 0
                rs_j_m2 = 0.0
                rain_mm_day = 0.0

                # reset rain state for next day too
                prev_rain_raw = None
                rain_mode_inferred = "cumulative"

        except Exception as e:
            tz_err = ZoneInfo(LOCAL_TZ)
            now_err = datetime.now(tz_err)
            print(f"[{now_err.strftime('%Y-%m-%d %H:%M:%S')}] ERROR: {e}")

        time.sleep(PY_LOOP_SEC)

if __name__ == "__main__":
    main()


Running... (minute logs; daily decision at 19:00; sends command to ThingSpeak)
[2026-01-17 17:33:03] T=12.73C RH=77.2% Rs=24.2W/m2 u2=0.15m/s RainRaw=0.500mm RainAdd=0.500mm Mode=cumulative dt=600s | Samples=1 Rs_cum=0.015MJ/m2 Rain_cum=0.500mm
[2026-01-17 17:34:03] T=12.62C RH=77.6% Rs=23.7W/m2 u2=0.15m/s RainRaw=0.500mm RainAdd=0.000mm Mode=cumulative dt=60s | Samples=2 Rs_cum=0.016MJ/m2 Rain_cum=0.500mm
